In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
Path.cwd()

PosixPath('/Users/mell/predicting-predictability/data')

In [6]:
sorted(p.name for p in Path.cwd().iterdir())

['.DS_Store',
 '.ipynb_checkpoints',
 'LINCS_small_molecules.tsv',
 'descriptor_summary_statistics.csv',
 'drug_master.json',
 'example_morgan_fingerprints.csv',
 'fingerprint_similarity_correlations.csv',
 'fingerprint_similarity_pairwise_sample.csv',
 'fingerprint_similarity_summary.csv',
 'lincs_morgan_fingerprints_2048.npy',
 'lincs_small_molecules_valid_unique.tsv',
 'morgan_autoencoder_bottleneck_sweep_results.csv',
 'morgan_autoencoder_latent15.csv',
 'morgan_autoencoder_latent15_umap.csv',
 'morgan_autoencoder_latent64_vectors.csv',
 'morgan_full_50_epoch_bottleneck_summary.csv',
 'morgan_full_50_epoch_loss_curves.csv',
 'morgan_to_maccs_latent32_results_summary.csv',
 'morgan_to_maccs_summary.md',
 'objective_3_lincs_morgan_autoencoder.ipynb',
 'part4_predicting.ipynb',
 'pca_cactvs_pubchem_fingerprints.csv',
 'pca_maccs_keys.csv',
 'pca_molecular_descriptors.csv',
 'pca_morgan_fingerprints.csv',
 'perturbseqr_cactvs_pubchem_fingerprints.csv',
 'perturbseqr_maccs_keys.csv',
 '

In [7]:
drug_master_path = Path.cwd() / "drug_master.json"

with open(drug_master_path, "r") as f:
    drug_master = json.load(f)

type(drug_master), len(drug_master)

(dict, 6801)

In [8]:
first_key = next(iter(drug_master))
first_key, drug_master[first_key]

('be-2254',
 {'name': 'be-2254',
  'pubchem_cid': '86308637',
  'inchi_key': 'PZZOEXPDTYIBPI-MRXNPFEDSA-N',
  'smiles': 'Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|',
  'broad_id': 'BRD-A24429032-003-03-2',
  'moa': 'adrenergic receptor antagonist',
  'target': 'ADRA1A',
  'clinical_phase': 'Phase 2',
  'disease_area': '',
  'indication': '',
  'fda_approved': False})

In [10]:
drug_master_df = pd.DataFrame.from_dict(drug_master, orient="index")
drug_master_df.index.name = "perturbseqr_drug_id"

print(drug_master_df.shape)
print(drug_master_df.columns.tolist())

drug_master_df[
    ["name", "smiles", "broad_id", "pubchem_cid", "inchi_key", "target", "moa"]
].isna().sum().sort_values()

(6801, 11)
['name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved']


name           0
smiles         0
broad_id       0
pubchem_cid    0
inchi_key      0
target         0
moa            0
dtype: int64

In [11]:
print("Unique Broad IDs:", drug_master_df["broad_id"].nunique())
print("Duplicate Broad IDs:", drug_master_df["broad_id"].duplicated().sum())

drug_master_df["broad_id"].value_counts().head(10)

Unique Broad IDs: 6777
Duplicate Broad IDs: 24


broad_id
                          25
BRD-A24429032-003-03-2     1
BRD-K62971431-001-01-9     1
BRD-K00003374-001-01-9     1
BRD-A66419424-001-02-4     1
BRD-K75527158-360-05-6     1
BRD-K20738689-001-01-2     1
BRD-K06753942-001-14-5     1
BRD-K00003398-001-01-9     1
BRD-K00003291-001-01-9     1
Name: count, dtype: int64

In [12]:
duplicate_broad_ids = (
    drug_master_df["broad_id"]
    .value_counts()
    .loc[lambda s: s > 1]
)

duplicate_broad_ids

broad_id
    25
Name: count, dtype: int64

In [13]:
shared_broad_id = duplicate_broad_ids.index[0]

drug_master_df.loc[
    drug_master_df["broad_id"] == shared_broad_id,
    ["name", "smiles", "broad_id", "pubchem_cid", "moa", "target"]
]

,name,smiles,broad_id,pubchem_cid,moa,target
perturbseqr_drug_id,,,,,,
terreic-acid-(-),terreic-acid-(-),,,,Bruton's tyrosine kinase (BTK) inhibitor,BTK
elactocin,elactocin,,,,exportin antagonist,XPO1
homochlorcyclizine,homochlorcyclizine,,,,antihistamine,HRH1
lcz696,lcz696,,,,angiotensin receptor antagonist,
fluorometholone,fluorometholone,,,,glucocorticoid receptor agonist,NR3C1
merck60,merck60,,,,HDAC inhibitor,HDAC1|HDAC2
diflorasone-diacetate,diflorasone-diacetate,,,,glucocorticoid receptor agonist,NR3C1
fpa-124,fpa-124,,,,AKT inhibitor,
rs-0481,rs-0481,,,,immunostimulant,


In [14]:
consensus_path = Path.cwd() / "cp_mean_coeff_mat.tsv.gz"

consensus_path.exists(), consensus_path.stat().st_size / 1e9

(True, 1.897525732)

In [15]:
preview = pd.read_csv(
    consensus_path,
    sep="\t",
    compression="gzip",
    nrows=5
)

preview.shape

(5, 12328)

In [16]:
preview.iloc[:, :8]

,Unnamed: 0,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC
0,afatinib,0.001353,-0.002113,-0.000360,0.000507,-0.000123,0.000799,-0.001517
1,erlotinib,0.000755,-0.000268,-0.000690,0.000107,0.000050,-0.000775,-0.000117
2,neratinib,0.000463,-0.003915,-0.000118,0.000174,-0.000900,0.000787,-0.001803
3,lapatinib,-0.000138,0.000255,0.000192,-0.000241,0.000348,0.000737,0.000374
4,pazopanib,-0.000276,0.000854,-0.000363,-0.000167,-0.000008,-0.001390,-0.000128


In [17]:
print("Number of gene columns:", preview.shape[1] - 1)
print("First column name:", preview.columns[0])
print("First five compound names:")
print(preview.iloc[:, 0].tolist())

Number of gene columns: 12327
First column name: Unnamed: 0
First five compound names:
['afatinib', 'erlotinib', 'neratinib', 'lapatinib', 'pazopanib']


In [18]:
# Read only the compound-name column from the full file
consensus_names = pd.read_csv(
    consensus_path,
    sep="\t",
    compression="gzip",
    usecols=[0]
)

consensus_names.columns = ["signature_name"]

print(consensus_names.shape)
print("Unique signature names:", consensus_names["signature_name"].nunique())
consensus_names.head()

(33609, 1)
Unique signature names: 33608


,signature_name
0,afatinib
1,erlotinib
2,neratinib
3,lapatinib
4,pazopanib


In [19]:
drug_names = set(drug_master_df["name"].str.lower().str.strip())
signature_names = set(consensus_names["signature_name"].str.lower().str.strip())

overlap = drug_names & signature_names

print("Drug-master compounds:", len(drug_names))
print("Consensus signature names:", len(signature_names))
print("Exact name overlap:", len(overlap))

Drug-master compounds: 6801
Consensus signature names: 33609
Exact name overlap: 3154


In [20]:
drug_master_df_clean = (
    drug_master_df
    .reset_index()
    .assign(match_name=lambda df: df["name"].str.lower().str.strip())
)

consensus_names_clean = (
    consensus_names
    .assign(match_name=lambda df: df["signature_name"].str.lower().str.strip())
)

exact_matches = drug_master_df_clean.merge(
    consensus_names_clean,
    on="match_name",
    how="inner"
)

exact_matches.shape

(3154, 14)

In [21]:
exact_matches[
    [
        "perturbseqr_drug_id",
        "name",
        "broad_id",
        "smiles",
        "signature_name",
        "moa",
        "target",
    ]
].head(10)

,perturbseqr_drug_id,name,broad_id,smiles,signature_name,moa,target
0,be-2254,be-2254,BRD-A24429032-003-03-2,Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|,BE-2254,adrenergic receptor antagonist,ADRA1A
1,dazmegrel,dazmegrel,BRD-K20738689-001-01-2,Cc1c(Cn2ccnc2)c2ccccc2n1CCC(O)=O,dazmegrel,thromboxane synthase inhibitor,TBXAS1
2,nobiletin,nobiletin,BRD-K06753942-001-14-5,COc1ccc(cc1OC)-c1cc(=O)c2c(OC)c(OC)c(OC)c(OC)c2o1,nobiletin,MEK inhibitor,MAP2K1
3,l-368899,l-368899,BRD-K00004282-003-01-9,Cc1ccccc1N1CCN(CC1)S(=O)(=O)C[C@]12CC[C@H](C[C...,L-368899,oxytocin receptor antagonist,AVPR1A|AVPR2|OXTR
4,betamethasone-valerate,betamethasone-valerate,BRD-K34032314-001-04-1,CCCCC(=O)O[C@@]1([C@@H](C)C[C@H]2[C@@H]3CCC4=C...,betamethasone-valerate,glucocorticoid receptor agonist,NR3C1
5,pnu-282987,pnu-282987,BRD-K28863208-001-03-8,Clc1ccc(cc1)C(=O)N[C@H]1CN2CCC1CC2,PNU-282987,cholinergic receptor agonist,CHRNA7
6,diclofenac,diclofenac,BRD-K08252256-236-33-8,OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl,diclofenac,cyclooxygenase inhibitor,AKR1C3|ALOX5|ASIC1|ASIC3|KCNQ2|KCNQ3|PLA2G2A|P...
7,hemado,hemado,BRD-K34672903-001-02-5,CCCCC#Cc1nc(NC)c2ncn([C@@H]3O[C@H](CO)[C@@H](O...,hemado,adenosine receptor agonist,ADORA1|ADORA2A|ADORA2B|ADORA3
8,proadifen,proadifen,BRD-K46317332-003-16-0,CCCC(C(=O)OCCN(CC)CC)(c1ccccc1)c1ccccc1,proadifen,nitric oxide synthase inhibitor,NOS1
9,simvastatin,simvastatin,BRD-K22134346-001-28-0,CCC(C)(C)C(=O)O[C@H]1C[C@@H](C)C=C2C=C[C@H](C)...,simvastatin,HMGCR inhibitor,HMGCR|ITGB2


In [22]:
exact_matches.to_csv(
    Path.cwd() / "matched_drugmaster_consensus_names.csv",
    index=False
)

print("Saved:", Path.cwd() / "matched_drugmaster_consensus_names.csv")

Saved: /Users/mell/predicting-predictability/data/matched_drugmaster_consensus_names.csv


In [23]:
morgan_preview = pd.read_csv(
    Path.cwd() / "perturbseqr_morgan_fingerprints.csv",
    nrows=3
)

morgan_preview.columns[:10], morgan_preview.shape

(Index(['record_id', 'name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id',
        'moa', 'target', 'clinical_phase', 'disease_area'],
       dtype='str'),
 (3, 2060))

In [24]:
morgan_preview.columns[-10:]

Index(['morgan_2038', 'morgan_2039', 'morgan_2040', 'morgan_2041',
       'morgan_2042', 'morgan_2043', 'morgan_2044', 'morgan_2045',
       'morgan_2046', 'morgan_2047'],
      dtype='str')

In [25]:
with open(Path.cwd() / "perturbseqr_morgan_fingerprints.csv", "r") as f:
    header = f.readline().strip().split(",")

print("Total columns:", len(header))
print("First metadata columns:", header[:12])
print("Last fingerprint columns:", header[-10:])

Total columns: 2060
First metadata columns: ['record_id', 'name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved']
Last fingerprint columns: ['morgan_2038', 'morgan_2039', 'morgan_2040', 'morgan_2041', 'morgan_2042', 'morgan_2043', 'morgan_2044', 'morgan_2045', 'morgan_2046', 'morgan_2047']


In [26]:
# Load just the metadata columns from the full Morgan file
morgan_metadata = pd.read_csv(
    Path.cwd() / "perturbseqr_morgan_fingerprints.csv",
    usecols=[
        "record_id", "name", "pubchem_cid", "inchi_key", "smiles",
        "broad_id", "moa", "target", "clinical_phase",
        "disease_area", "indication", "fda_approved"
    ]
)

morgan_metadata.shape

(6754, 12)

In [27]:
morgan_metadata["match_name"] = morgan_metadata["name"].str.lower().str.strip()

matched_morgan_metadata = morgan_metadata.merge(
    exact_matches[
        ["perturbseqr_drug_id", "match_name", "signature_name"]
    ],
    on="match_name",
    how="inner"
)

matched_morgan_metadata.shape

(3131, 15)

In [28]:
exact_match_ids = set(exact_matches["perturbseqr_drug_id"])
morgan_match_ids = set(matched_morgan_metadata["perturbseqr_drug_id"])

missing_from_morgan = (
    exact_matches[
        ~exact_matches["perturbseqr_drug_id"].isin(morgan_match_ids)
    ][
        ["perturbseqr_drug_id", "name", "smiles", "broad_id", "signature_name"]
    ]
)

print("Exact matches missing from Morgan feature file:", len(missing_from_morgan))
missing_from_morgan.head(20)

Exact matches missing from Morgan feature file: 23


,perturbseqr_drug_id,name,smiles,broad_id,signature_name
299,homochlorcyclizine,homochlorcyclizine,,,homochlorcyclizine
408,atropine,atropine,CN1[C@H]2CC[C@@H]1C[C@@H](C2)OC(=O)[C@@H](CO)c...,BRD-K01825666-330-02-9,atropine
503,merck60,merck60,,,Merck60
600,ipratropium,ipratropium,CC(C)[N@@+]1(C)[C@H]2CC[C@@H]1C[C@@H](C2)OC(=O...,BRD-K01826778-004-06-9,ipratropium
655,alcuronium,alcuronium,OC\C=C1/C[N@+]2(CC=C)CC[C@@]34[C@@H]2C[C@@H]1\...,BRD-A84907376-300-03-8,alcuronium
739,ub-165,ub-165,"Clc1ccc(cn1)C1=CCC[C@H]2CC[C@@H]1N2 |&1:11,&2:...",BRD-A14574269-051-03-0,UB-165
797,deptropine,deptropine,CN1[C@H]2CC[C@@H]1C[C@@H](C2)OC1c2ccccc2CCc2cc...,BRD-K00005351-050-01-9,deptropine
1295,homatropine,homatropine,CN1[C@H]2CC[C@@H]1C[C@@H](C2)OC(=O)[C@@H](O)c1...,BRD-K01826529-004-11-9,homatropine
1382,zacopride,zacopride,COc1cc(N)c(Cl)cc1C(=O)N[C@H]1CN2CCC1CC2 |&1:13...,BRD-K01826799-003-03-9,zacopride
1567,estramustine,estramustine,,,estramustine


In [29]:
feature_files = {
    "MACCS": "perturbseqr_maccs_keys.csv",
    "CACTVS": "perturbseqr_cactvs_pubchem_fingerprints.csv",
    "Descriptors": "perturbseqr_molecular_descriptors.csv",
}

for label, filename in feature_files.items():
    df = pd.read_csv(Path.cwd() / filename, usecols=["record_id", "name"])
    print(label, df.shape, (df["record_id"] == morgan_metadata["record_id"]).all())

MACCS (6754, 2) True


ValueError: Usecols do not match columns, columns expected but not found: ['record_id']

In [30]:
for label, filename in feature_files.items():
    preview_df = pd.read_csv(Path.cwd() / filename, nrows=2)
    print(f"\n{label}")
    print("Shape:", preview_df.shape)
    print("First 15 columns:", preview_df.columns[:15].tolist())


MACCS
Shape: (2, 179)
First 15 columns: ['record_id', 'name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved', 'maccs_0', 'maccs_1', 'maccs_2']


OverflowError: int too large to convert to float

In [31]:
import csv

for label, filename in feature_files.items():
    with open(Path.cwd() / filename, "r", newline="") as f:
        header = next(csv.reader(f))

    print(f"\n{label}")
    print("Total columns:", len(header))
    print("First 15 columns:", header[:15])
    print("Last 10 columns:", header[-10:])


MACCS
Total columns: 179
First 15 columns: ['record_id', 'name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved', 'maccs_0', 'maccs_1', 'maccs_2']
Last 10 columns: ['maccs_157', 'maccs_158', 'maccs_159', 'maccs_160', 'maccs_161', 'maccs_162', 'maccs_163', 'maccs_164', 'maccs_165', 'maccs_166']

CACTVS
Total columns: 893
First 15 columns: ['name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved', 'cactvs_pubchem_fp', 'cactvs_0', 'cactvs_1', 'cactvs_2']
Last 10 columns: ['cactvs_871', 'cactvs_872', 'cactvs_873', 'cactvs_874', 'cactvs_875', 'cactvs_876', 'cactvs_877', 'cactvs_878', 'cactvs_879', 'cactvs_880']

Descriptors
Total columns: 21
First 15 columns: ['record_id', 'name', 'pubchem_cid', 'inchi_key', 'smiles', 'broad_id', 'moa', 'target', 'clinical_phase', 'disease_area', 'indication', 'fda_approved', 'molecular_weight', 

In [32]:
# Check whether matched names map uniquely to one SMILES string
matched_morgan_metadata.groupby("match_name")["smiles"].nunique().value_counts()

smiles
1    3131
Name: count, dtype: int64

In [33]:
model_index = matched_morgan_metadata[
    [
        "record_id",
        "perturbseqr_drug_id",
        "name",
        "match_name",
        "smiles",
        "broad_id",
        "signature_name",
        "moa",
        "target",
    ]
].copy()

model_index.to_csv(
    Path.cwd() / "part4_model_compound_index.csv",
    index=False
)

model_index.shape, model_index.head()

((3131, 9),
                 record_id     perturbseqr_drug_id                    name  \
 0                 be-2254                 be-2254                 be-2254   
 1               dazmegrel               dazmegrel               dazmegrel   
 2               nobiletin               nobiletin               nobiletin   
 3                l-368899                l-368899                l-368899   
 4  betamethasone-valerate  betamethasone-valerate  betamethasone-valerate   
 
                match_name                                             smiles  \
 0                 be-2254            Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|   
 1               dazmegrel                   Cc1c(Cn2ccnc2)c2ccccc2n1CCC(O)=O   
 2               nobiletin  COc1ccc(cc1OC)-c1cc(=O)c2c(OC)c(OC)c(OC)c(OC)c2o1   
 3                l-368899  Cc1ccccc1N1CCN(CC1)S(=O)(=O)C[C@]12CC[C@H](C[C...   
 4  betamethasone-valerate  CCCCC(=O)O[C@@]1([C@@H](C)C[C@H]2[C@@H]3CCC4=C...   
 
                  broad_id    

In [34]:
# Names we want from the consensus-signature file
wanted_signatures = set(model_index["signature_name"].str.lower().str.strip())

matched_signature_chunks = []

for chunk in pd.read_csv(
    consensus_path,
    sep="\t",
    compression="gzip",
    chunksize=500,
):
    chunk = chunk.rename(columns={chunk.columns[0]: "signature_name"})
    chunk["match_name"] = chunk["signature_name"].str.lower().str.strip()

    keep = chunk["match_name"].isin(wanted_signatures)
    if keep.any():
        matched_signature_chunks.append(chunk.loc[keep])

matched_signatures = pd.concat(matched_signature_chunks, ignore_index=True)

matched_signatures.shape

/var/folders/fn/ctr407bj6j96h5nhl6snwxqr0000gn/T/ipykernel_7622/3319732139.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk["match_name"] = chunk["signature_name"].str.lower().str.strip()


(3131, 12329)

In [35]:
matched_signatures[["signature_name", "match_name"]].head()

,signature_name,match_name
0,afatinib,afatinib
1,erlotinib,erlotinib
2,neratinib,neratinib
3,lapatinib,lapatinib
4,pazopanib,pazopanib


In [37]:
matched_signatures.to_csv(
    Path.cwd() / "part4_matched_consensus_signatures.csv.gz",
    index=False,
    compression="gzip"
)

print("Saved compressed CSV target matrix.")

Saved compressed CSV target matrix.
